# 第 60 课｜重要性采样、概率比与 PPO-Clip

上一课每次采样后只更新一次。本课保存**采样时的旧概率**，在明确预算内重复优化同一批回答，并研究怎样防止过度利用旧数据。

**完成目标：** 能手算重要性比与正负优势的裁剪；区分当前策略、行为旧策略与固定参考；实现有边界的复用循环；用独立结果判断训练是否值得保留。

**范围声明：** 数学实验＋实际的小型字符 Transformer、Q/V LoRA 训练。主实验把完整回答看作一个动作，使用**序列级 clipped surrogate 和固定留一法优势**，不是完整逐 token PPO，也没有 critic、GAE、value loss 或 GRPO。不是预训练大模型成绩。

## 0. 环境与学习方式

使用现有 PyTorch 环境，Python 3.10+。CPU 足够，不下载模型或语料。请从全新内核按顺序运行；讲义本身已经包含全部实现，不需要配套 `.py`。

先预测公式输出，再运行。第一次可将 `RUN_TRAINING=False` 跳过完整训练，数学实验和自动检查不受影响；默认值为 True。独立练习检查默认关闭，全部单元运行成功**不等于已经完成练习**。

脚本入口：
```bash
python lesson_60_importance_sampling_and_ppo_clip.py --checks-only
python lesson_60_importance_sampling_and_ppo_clip.py --math-only
python lesson_60_importance_sampling_and_ppo_clip.py
```

文件写入当前工作目录下 `lesson_60_outputs/run_时间戳/`，不覆盖旧实验。明确请求 CUDA 不可用就报错，不静默回退。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第60课：重要性采样、概率比与PPO-Clip。

Python 3.10+，PyTorch；完全离线。实际训练小型字符Transformer的Q/V LoRA。
完整回答视为一个动作，使用序列级clipped surrogate与固定LOO优势；
没有critic/GAE/PPO value loss，不冒充完整token-level PPO或GRPO。
"""
from __future__ import annotations

# %% 01 环境、约定与配置
import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import tempfile
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境；请在同一环境选择Jupyter内核。") from exc

VERSION = "lesson60-v1"
PAD, BOS, EOS, IGNORE = 0, 1, 2, -100
SOURCES = {
    "ppo_paper": "https://arxiv.org/abs/1707.06347",
    "ppo_explanation": "https://spinningup.openai.com/en/latest/algorithms/ppo.html",
    "ppo_implementation": "https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html",
    "autograd": "https://docs.pytorch.org/docs/2.10/notes/autograd.html",
    "detach": "https://docs.pytorch.org/docs/2.10/generated/torch.Tensor.detach.html",
    "reproducibility": "https://docs.pytorch.org/docs/2.10/notes/randomness.html",
}

class LessonError(ValueError):
    """不满足本课约定时明确拒绝，不替换目标或伪造奖励。"""

def require(ok: bool, message: str) -> None:
    if not ok:
        raise LessonError(message)

def canonical(value) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)

def digest(value) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()

def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True,
                              indent=2, allow_nan=False) + "\n", encoding="utf-8")

def tensor_hash(values: dict[str, torch.Tensor]) -> str:
    h = hashlib.sha256()
    for name, tensor in sorted(values.items()):
        t = tensor.detach().cpu().contiguous()
        h.update(canonical([name, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()

def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def resolve_device(name: str) -> torch.device:
    require(name in {"cpu", "cuda", "auto"}, "device只支持cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    require(name != "cuda" or torch.cuda.is_available(), "CUDA不可用，不会静默回退。")
    return torch.device(name)

@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int = 20
    dim: int = 32
    heads: int = 4
    layers: int = 2
    ff_dim: int = 64
    max_len: int = 24

    def validate(self) -> None:
        require(all(type(x) is int and x > 0 for x in asdict(self).values()), "配置必须为正整数")
        require(self.dim % self.heads == 0, "dim需要整除heads")
        require(self.vocab_size > 3 and self.max_len >= 16, "词表/上下文过小")

@dataclass(frozen=True)
class TrainConfig:
    seed: int = 60
    sft_steps: int = 40
    sft_lr: float = 0.003
    rollout_batches: int = 24
    prompts_per_update: int = 8
    group_size: int = 4
    inner_epochs: int = 4
    rl_lr: float = 0.003
    clip_epsilon: float = 0.2
    target_state_kl: float = 0.02
    rank: int = 4
    alpha: float = 8.0
    max_grad_norm: float = 1.0
    eval_samples: int = 8
    max_new_tokens: int = 3

    def validate(self):
        for key in ("sft_steps", "rollout_batches", "prompts_per_update", "group_size",
                    "inner_epochs", "rank", "eval_samples"):
            value = getattr(self, key)
            require(type(value) is int and value > 0, key+"必须是正整数")
        require(type(self.seed) is int and self.seed >= 0, "seed非法")
        require(self.group_size >= 2, "LOO需要G>=2")
        require(self.max_new_tokens == 3, "任务固定3步回答视野")
        require(type(self.clip_epsilon) in (int,float) and math.isfinite(self.clip_epsilon)
                and 0 < self.clip_epsilon < 1, "epsilon必须在(0,1)")
        for key in ("sft_lr", "rl_lr", "target_state_kl", "alpha", "max_grad_norm"):
            value = getattr(self,key)
            require(type(value) in (int,float) and math.isfinite(value) and value > 0,
                    key+"必须有限正数")

In [2]:
torch.set_num_threads(1)
DEVICE = "cpu"
OUTPUT_ROOT = Path("lesson_60_outputs")
TRAIN_CONFIG = TrainConfig()
RUN_TRAINING = True
seed_all(TRAIN_CONFIG.seed)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "CUDA可用:", torch.cuda.is_available())
print("设备:", DEVICE)
print("训练配置:", asdict(TRAIN_CONFIG))

Python: 3.13.5
PyTorch: 2.10.0+cpu CUDA可用: False
设备: cpu
训练配置: {'seed': 60, 'sft_steps': 40, 'sft_lr': 0.003, 'rollout_batches': 24, 'prompts_per_update': 8, 'group_size': 4, 'inner_epochs': 4, 'rl_lr': 0.003, 'clip_epsilon': 0.2, 'target_state_kl': 0.02, 'rank': 4, 'alpha': 8.0, 'max_grad_norm': 1.0, 'eval_samples': 8, 'max_new_tokens': 3}


## 1. 三个策略，三个不同问题

| 名称 | 在什么时候变化 | 用在哪里 |
|---|---|---|
| 当前策略 $\pi_\theta$ | 每次 optimizer.step 后可能变化 | 当前回答概率与梯度 |
| 行为旧策略 $\pi_{old}$ | 每批重新采样时更新；同一批内固定 | 概率比的分母 |
| 固定参考 $\pi_{ref}$ | 本课从 SFT 完成后始终固定 | 只监测相对起点的漂移 |

PPO 会交替采样与优化，每批允许多轮局部更新；它不是任意复用历史数据的通用离策略保证。[PPO 原论文](https://arxiv.org/abs/1707.06347)

本课**不在目标中加入固定参考 KL 惩罚**，避免把概率比裁剪、旧策略 KL 停止和参考 KL 正则混为一谈。

假设参考为 `[0.5,0.5]`、旧策略为 `[0.8,0.2]`、当前为 `[0.9,0.1]`。先手算：概率比的正确分母是谁？

In [3]:
ref_demo = torch.tensor([0.5, 0.5], dtype=torch.float64)
old_demo = torch.tensor([0.8, 0.2], dtype=torch.float64)
new_demo = torch.tensor([0.9, 0.1], dtype=torch.float64)
print("正确 current / old:", (new_demo / old_demo).tolist())
print("错误地用 reference:", (new_demo / ref_demo).tolist())

正确 current / old: [1.125, 0.5]
错误地用 reference: [1.8, 0.2]


## 2. 固定任务与动作空间

任务仍是复制末尾两位数字：`copy:03,12=` → `12` 加 EOS。

两个数来自 00–15，共 256 个提示，分为 160 训练、48 开发、48 测试。答案动作只包括十个数字与 EOS，温度为 1，不做 Top-k/Top-p。每条回答最多三个 token，出现 EOS 后结束。

这是**固定视野下的受限策略**。精确正确且正常 EOS 奖 1，其他合法任务结果奖 0；运行异常不是低奖励样本，而是中止实验。参考答案仅用于评分，不进入生成输入。

三步上限是任务终止规则，故不需要对截断后未知回报作 value bootstrap。它不同于把本来更长的任务任意切断。

In [4]:
# %% 02 固定复制任务：数据、标签与精确奖励
class CharCodec:
    """固定复制任务的字符表。回答策略只允许十个数字及EOS。"""
    def __init__(self):
        self.tokens = ["<pad>", "<bos>", "<eos>"] + list("copy:0123456789,=")
        require(len(set(self.tokens)) == len(self.tokens), "词表不能重复")
        self.to_id = {s: i for i,s in enumerate(self.tokens)}
        self.answer_ids = [self.to_id[str(i)] for i in range(10)] + [EOS]

    def encode(self, text: str) -> list[int]:
        require(type(text) is str and all(c in self.to_id for c in text), "未知字符")
        return [self.to_id[c] for c in text]

    def decode_answer(self, ids: list[int]) -> str:
        require(all(type(i) is int and i in self.answer_ids and i != EOS for i in ids),
                "正文必须是数字token；EOS应在外层处理")
        return "".join(self.tokens[i] for i in ids)

    def contract(self) -> dict:
        return {"tokens": self.tokens, "answer_ids": self.answer_ids,
                "sampling": "categorical_with_replacement_temperature_1_no_topk_no_topp",
                "horizon": 3, "end": "first_EOS_or_cap", "version": VERSION}

def make_splits(seed: int = 60) -> dict[str, list[dict]]:
    """复制末尾两位十进制串；共享语法/数值范围，不共享完整prompt。"""
    rows = [{"id": f"copy_{a:02d}_{b:02d}", "prompt": f"copy:{a:02d},{b:02d}=",
             "answer": f"{b:02d}", "source": f"{a:02d},{b:02d}"}
            for a in range(16) for b in range(16)]
    random.Random(seed).shuffle(rows)
    out = {"train": rows[:160], "dev": rows[160:208], "test": rows[208:]}
    for name, group in out.items():
        for row in group:
            row["split"] = name
    return out

def validate_splits(splits: dict) -> None:
    require(set(splits) == {"train","dev","test"}, "划分名称错误")
    seen = set()
    for name,rows in splits.items():
        require(bool(rows), "划分不能为空")
        for row in rows:
            require(row["split"] == name, "划分标签错误")
            require(row["prompt"] not in seen, "prompt跨侧或重复")
            require(row["answer"] == row["prompt"][-3:-1], "任务标签与显式规则冲突")
            seen.add(row["prompt"])

def score_completion(text: str, stop_reason: str, expected: str) -> dict:
    """完成EOS且精确匹配两位串才奖1；cap/格式/错误答案均按本任务定义奖0。
    推理异常、数据错误不在此伪造为0奖励：它们抛出异常并中止更新。
    """
    require(type(text) is str and type(expected) is str, "评分字段必须是字符串")
    require(len(expected)==2 and expected.isascii() and expected.isdigit(), "参考答案必须两位数字")
    require(stop_reason in {"eos", "max_tokens"}, "执行故障不是低奖励样本")
    if stop_reason != "eos":
        return {"reward": 0.0, "reason": "missing_eos"}
    if len(text) != 2 or not text.isascii() or not text.isdigit():
        return {"reward": 0.0, "reason": "invalid_format"}
    return {"reward": float(text == expected), "reason": "correct" if text==expected else "wrong"}

def collate_completions(rows: list[dict], answers: list[list[int]], codec: CharCodec,
                        max_len: int, device="cpu") -> dict:
    require(bool(rows) and len(rows)==len(answers), "批次与回答不匹配")
    seqs, targets = [], []
    for row, ans in zip(rows, answers):
        require(bool(ans) and len(ans) <= 3, "回答长度需要1..3")
        require(all(type(t) is int and t in codec.answer_ids for t in ans), "回答token非法")
        require(EOS not in ans[:-1], "EOS后不能再有token")
        prefix = [BOS] + codec.encode(row["prompt"])
        require(len(prefix)+len(ans)<=max_len, "超出上下文；不静默截断")
        seqs.append(prefix+list(ans))
        targets.append([IGNORE]*len(prefix)+list(ans))
    width = max(map(len, seqs))
    ids = torch.tensor([s+[PAD]*(width-len(s)) for s in seqs], dtype=torch.long, device=device)
    labels = torch.tensor([s+[IGNORE]*(width-len(s)) for s in targets], dtype=torch.long, device=device)
    return {"input_ids":ids, "labels":labels}

In [5]:
codec = CharCodec()
splits = make_splits(60)
validate_splits(splits)
print("划分:", {k:len(v) for k,v in splits.items()})
print("动作:", [codec.tokens[i] for i in codec.answer_ids])
print("样例:", splits["train"][0])
for text, stop in [("12","eos"),("12","max_tokens"),("2","eos")]:
    print(text, stop, score_completion(text,stop,"12"))

划分: {'train': 160, 'dev': 48, 'test': 48}
动作: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '<eos>']
样例: {'id': 'copy_09_11', 'prompt': 'copy:09,11=', 'answer': '11', 'source': '09,11', 'split': 'train'}
12 eos {'reward': 1.0, 'reason': 'correct'}
12 max_tokens {'reward': 0.0, 'reason': 'missing_eos'}
2 eos {'reward': 0.0, 'reason': 'invalid_format'}


## 3. 小型模型与冻结基座

两层、隐藏维度 32、4 头的因果 Transformer，全部 Dropout 为零。先进行 SFT 预热，再冻结基座，仅训练每层 Q/V 的 rank=4 LoRA。

代码保留了上一课的基础结构，但完整复制进本课，避免依赖其他文件。`eval()` 不关闭自动求导；有梯度的概率重算不能放进 `no_grad()`。[PyTorch autograd](https://docs.pytorch.org/docs/2.10/notes/autograd.html)

In [6]:
# %% 03 自回归Transformer与Q/V LoRA；复制基础实现以便独立运行
class CausalAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.heads, self.head_dim = cfg.heads, cfg.dim // cfg.heads
        self.q_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.k_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.v_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)
        self.o_proj = nn.Linear(cfg.dim, cfg.dim, bias=False)

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        def heads(z):
            return z.reshape(b, t, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = (heads(layer(x)) for layer in [self.q_proj, self.k_proj, self.v_proj])
        causal = torch.ones(t, t, dtype=torch.bool, device=x.device).tril()
        allowed = causal[None, None, :, :] & valid[:, None, None, :]
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weights = F.softmax(scores.masked_fill(~allowed, -torch.inf), dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().reshape(b, t, d)
        return self.o_proj(out)

class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.dim)
        self.attn = CausalAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.dim)
        self.ffn = nn.Sequential(nn.Linear(cfg.dim, cfg.ff_dim), nn.GELU(),
                                 nn.Linear(cfg.ff_dim, cfg.dim))

    def forward(self, x: torch.Tensor, valid: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), valid)
        return x + self.ffn(self.norm2(x))

class TinyLM(nn.Module):
    """小型因果Transformer，绝对位置、Pre-LN、GELU；所有Dropout均为零。

    为聚焦策略梯度故意不添加RoPE、GQA、KV cache或量化；不是生产模型缩小复制品。
    forward只返回logits，由外部损失移位一次。输入必须从BOS开始、右侧补齐。
    """
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.dim, padding_idx=PAD)
        self.position_embedding = nn.Embedding(cfg.max_len, cfg.dim)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.layers)])
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        require(ids.ndim == 2 and ids.dtype == torch.long, "input_ids需要[B,T] long")
        require(ids.shape[0] > 0 and 0 < ids.shape[1] <= self.cfg.max_len, "输入长度非法")
        require(bool(((ids >= 0) & (ids < self.cfg.vocab_size)).all()), "token越界")
        require(bool((ids[:, 0] == BOS).all()), "本课每条输入从BOS开始，拒绝全PAD")
        valid = ids.ne(PAD)
        require(not bool((valid[:, 1:] & ~valid[:, :-1]).any()), "本课只支持右补齐")
        positions = torch.arange(ids.shape[1], device=ids.device)
        x = self.token_embedding(ids) + self.position_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x, valid)
        return self.head(self.norm(x))

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        require(type(base) is nn.Linear, "只接受未注入的Linear")
        require(type(rank) is int and 0 < rank <= min(base.in_features, base.out_features),
                "rank超出本课允许范围")
        require(math.isfinite(alpha) and alpha > 0, "alpha必须有限为正")
        self.base = base
        self.base.requires_grad_(False)
        self.rank, self.alpha, self.scaling = rank, float(alpha), float(alpha) / rank
        self.enabled = True
        self.lora_A = nn.Parameter(base.weight.new_empty(rank, base.in_features))
        self.lora_B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        if self.enabled:
            out = out + self.scaling * F.linear(F.linear(x, self.lora_A), self.lora_B)
        return out

def is_adapter(name: str) -> bool:
    return name.endswith(".lora_A") or name.endswith(".lora_B")

def inject_lora(model: TinyLM, rank: int = 4, alpha: float = 8.0) -> TinyLM:
    require(not any(isinstance(m, LoRALinear) for m in model.modules()), "拒绝重复注入")
    model.requires_grad_(False)
    # SFT刚结束可能留有.grad；清掉它们，才能审计RL阶段是否产生新梯度。
    for p in model.parameters():
        p.grad = None
    for block in model.blocks:
        block.attn.q_proj = LoRALinear(block.attn.q_proj, rank, alpha)
        block.attn.v_proj = LoRALinear(block.attn.v_proj, rank, alpha)
    require(all(p.requires_grad == is_adapter(n) for n, p in model.named_parameters()),
            "可训练集合必须恰好等于Q/V的A/B")
    return model

def freeze_reference(base: TinyLM) -> TinyLM:
    ref = copy.deepcopy(base).eval().requires_grad_(False)
    for p in ref.parameters():
        p.grad = None
    return ref

def base_state(model: TinyLM) -> dict[str, torch.Tensor]:
    return {name.replace(".base.", "."): value
            for name, value in model.state_dict().items() if not is_adapter(name)}

def adapter_snapshot(model: TinyLM) -> dict[str, torch.Tensor]:
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters()
            if is_adapter(name)}

def load_adapter(model: TinyLM, state: dict) -> None:
    expected = {n for n, _ in model.named_parameters() if is_adapter(n)}
    require(set(state) == expected, "adapter键集合不匹配")
    # 先检查全部张量，再修改，避免部分加载。
    params = dict(model.named_parameters())
    for name, value in state.items():
        require(isinstance(value, torch.Tensor) and value.shape == params[name].shape,
                "adapter形状不匹配")
        require(value.is_floating_point() and bool(torch.isfinite(value).all()), "adapter非法数值")
    with torch.no_grad():
        for name, value in state.items():
            params[name].copy_(value.to(params[name]))

@contextmanager
def adapter_disabled(model: TinyLM):
    layers = [m for m in model.modules() if isinstance(m, LoRALinear)]
    previous = [m.enabled for m in layers]
    try:
        for layer in layers:
            layer.enabled = False
        yield
    finally:
        for layer, enabled in zip(layers, previous):
            layer.enabled = enabled

In [7]:
seed_all(60)
probe_base = TinyLM(ModelConfig())
probe_reference = freeze_reference(probe_base)
probe_policy = inject_lora(copy.deepcopy(probe_base)).eval()
print("基座参数:", sum(p.numel() for p in probe_base.parameters()))
print("LoRA可训练参数:", sum(p.numel() for p in probe_policy.parameters() if p.requires_grad))
print("参考全部冻结:", all(not p.requires_grad for p in probe_reference.parameters()))

基座参数: 18944
LoRA可训练参数: 1024
参考全部冻结: True


## 4. 采样与重算必须使用同一个分布

生成时不保存计算图；得到回答后，把相同提示和回答重新前向，计算带梯度的序列 log 概率。

本课同位 labels，在函数中移位一次；只累计回答与 EOS，排除提示和补齐。采样与重算都经过同一个 `answer_log_probs()`，不允许一个路径温度为 1，另一个路径过滤后再算。

每个回答的概率是各条件概率的乘积，所以 log 概率沿 token **求和**，不是求均值。

In [8]:
# %% 04 概率接口与固定动作集合
def answer_log_probs(logits: torch.Tensor, codec: CharCodec) -> torch.Tensor:
    """固定合法动作集合的归一化log概率；采样与重评分必须调用同一函数。"""
    require(logits.shape[-1] == len(codec.tokens), "logits词表宽度错误")
    require(bool(torch.isfinite(logits).all()), "模型产生非有限logits")
    return F.log_softmax(logits[..., codec.answer_ids], dim=-1)

def completion_logps(logits: torch.Tensor, labels: torch.Tensor,
                     codec: CharCodec) -> tuple[torch.Tensor, torch.Tensor]:
    require(logits.ndim==3 and labels.shape==logits.shape[:2], "logits/labels形状错误")
    require(labels.dtype==torch.long, "labels需要long")
    require(bool((labels[:,0]==IGNORE).all()), "首位置无前缀预测，必须忽略")
    target = labels[:,1:]
    mask = target.ne(IGNORE)
    require(bool((mask.sum(-1)>0).all()), "每条回答必须有监督/采样位置")
    mapping = labels.new_full((len(codec.tokens),), -1)
    mapping[codec.answer_ids] = torch.arange(len(codec.answer_ids),device=labels.device)
    safe = target.masked_fill(~mask, EOS)
    require(bool(((safe>=0)&(safe<len(codec.tokens))).all()), "标签越界")
    action_ids = mapping[safe]
    require(bool((action_ids>=0).all()), "目标不在固定回答动作集合")
    lp = answer_log_probs(logits[:,:-1,:], codec).gather(-1,action_ids.unsqueeze(-1)).squeeze(-1)
    sequence_lp = lp.masked_fill(~mask,0).sum(-1)
    return sequence_lp, mask.sum(-1)

def leave_one_out(values: torch.Tensor) -> torch.Tensor:
    require(values.ndim==2 and values.shape[1]>=2, "LOO输入需要[B,G]且G>=2")
    require(values.is_floating_point() and bool(torch.isfinite(values).all()), "奖励必须有限浮点")
    v = values.detach()
    baseline = (v.sum(dim=1,keepdim=True)-v)/(v.shape[1]-1)
    return v-baseline

@contextmanager
def eval_mode(model: nn.Module):
    old=model.training
    try:
        model.eval()
        yield
    finally:
        model.train(old)

@torch.no_grad()
def sample_rollouts(policy: TinyLM, rows: list[dict], codec: CharCodec,
                    group_size: int, generator: torch.Generator,
                    horizon: int = 3, greedy: bool = False) -> dict:
    require(bool(rows) and type(group_size) is int and group_size>0, "空输入/非法组大小")
    require(type(horizon) is int and 1<=horizon<=3, "horizon必须1..3")
    expanded=[copy.deepcopy(row) for row in rows for _ in range(group_size)]
    prefixes=[[BOS]+codec.encode(row["prompt"]) for row in expanded]
    require(len({len(x) for x in prefixes})==1, "本课采样提示固定等长，不支持左补齐混用")
    require(len(prefixes[0])+horizon<=policy.cfg.max_len,"生成预算超过上下文")
    device=next(policy.parameters()).device
    ids=torch.tensor(prefixes,dtype=torch.long,device=device)
    active=torch.ones(len(expanded),dtype=torch.bool,device=device)
    answers=[[] for _ in expanded]; behavior=torch.zeros(len(expanded),dtype=torch.float64,device=device)
    with eval_mode(policy):
        for _ in range(horizon):
            lp=answer_log_probs(policy(ids)[:,-1,:],codec)
            action=lp.argmax(-1) if greedy else torch.multinomial(lp.exp(),1,generator=generator).squeeze(1)
            token_table=torch.tensor(codec.answer_ids,device=device)
            next_ids=token_table[action]
            chosen_lp=lp.gather(1,action[:,None]).squeeze(1)
            behavior+=chosen_lp.to(torch.float64)*active
            old_active=active.clone()
            for i in range(len(expanded)):
                if bool(old_active[i]):
                    answers[i].append(int(next_ids[i]))
            active=active & next_ids.ne(EOS)
            # 已结束行追加PAD。仍活跃行的计算不受其他batch行影响。
            append=next_ids.masked_fill(~old_active,PAD)
            ids=torch.cat([ids,append[:,None]],dim=1)
            if not bool(active.any()):
                break
    samples=[]
    for i,(row,ans) in enumerate(zip(expanded,answers)):
        stopped=ans[-1]==EOS
        text=codec.decode_answer(ans[:-1] if stopped else ans)
        scoring=score_completion(text,"eos" if stopped else "max_tokens",row["answer"])
        samples.append({"task_id":row["id"],"group_index":i//group_size,"sample_index":i%group_size,
                        "prompt":row["prompt"],"token_ids":ans,"text":text,
                        "stop_reason":"eos" if stopped else "max_tokens",
                        "behavior_logp":float(behavior[i]),**scoring})
    return {"rows":expanded,"answers":answers,"samples":samples,
            "prompt_count":len(rows),"group_size":group_size,
            "policy_hash":tensor_hash(policy.state_dict()),"codec_hash":digest(codec.contract()),
            "horizon":horizon,"greedy":greedy,"used":False}

def sft_warmup(base: TinyLM, rows: list[dict], codec: CharCodec, cfg: TrainConfig) -> list[dict]:
    require(all(row["split"]=="train" for row in rows),"SFT只接收训练侧")
    optimizer=torch.optim.Adam(base.parameters(),lr=cfg.sft_lr)
    rng=random.Random(cfg.seed+1); history=[]
    device=next(base.parameters()).device
    base.train()
    for step in range(cfg.sft_steps):
        batch_rows=rng.sample(rows, min(32,len(rows)))
        answers=[codec.encode(r["answer"])+[EOS] for r in batch_rows]
        batch=collate_completions(batch_rows,answers,codec,base.cfg.max_len,device)
        optimizer.zero_grad(set_to_none=True)
        lp,counts=completion_logps(base(batch["input_ids"]),batch["labels"],codec)
        loss=-lp.sum()/counts.sum()
        loss.backward()
        nn.utils.clip_grad_norm_(base.parameters(),cfg.max_grad_norm,error_if_nonfinite=True)
        optimizer.step()
        if step==0 or (step+1)%10==0 or step+1==cfg.sft_steps:
            history.append({"step":step+1,"token_nll":float(loss.detach())})
    base.eval()
    for p in base.parameters(): p.grad=None
    return history

@torch.no_grad()
def evaluate_policy(model: TinyLM, rows: list[dict], codec: CharCodec,
                    sample_count: int=8, seed: int=90059) -> dict:
    require(bool(rows),"评测不能为空")
    device=next(model.parameters()).device
    answers=[codec.encode(r["answer"])+[EOS] for r in rows]
    batch=collate_completions(rows,answers,codec,model.cfg.max_len,device)
    with eval_mode(model):
        target_lp,_=completion_logps(model(batch["input_ids"]),batch["labels"],codec)
    # 本任务只有一种接受的完整轨迹(answer+EOS)，故exp(logp)就是它的精确成功概率。
    probabilities=target_lp.exp().cpu()
    gen=torch.Generator(device=device).manual_seed(seed)
    sampled=sample_rollouts(model,rows,codec,sample_count,gen)
    greedy=sample_rollouts(model,rows,codec,1,gen,greedy=True)
    return {"tasks":len(rows),"exact_success_probability":float(probabilities.mean()),
            "mean_correct_sequence_nll":float(-target_lp.mean()),
            "sampled_successes":sum(s["reward"] for s in sampled["samples"]),
            "sampled_total":len(sampled["samples"]),
            "sampled_success_rate":sum(s["reward"] for s in sampled["samples"])/len(sampled["samples"]),
            "greedy_successes":sum(s["reward"] for s in greedy["samples"]),
            "per_task_probability":[{"id":r["id"],"p":float(p)} for r,p in zip(rows,probabilities)],
            "greedy":greedy["samples"],"samples":sampled["samples"],
            "note":"小型同语法复制任务；概率针对本课固定动作集合和三步视野。"}

def compact_eval(report: dict) -> dict:
    return {k:v for k,v in report.items() if k not in {"per_task_probability","greedy","samples"}}

## 5. 重要性采样：旧分布的数据怎样估计新分布的期望？

对固定问题，若旧分布覆盖当前分布的支持集：

$$\mathbb E_{y\sim\pi_\theta}[R(y)]
=\mathbb E_{y\sim\pi_{old}}\left[\frac{\pi_\theta(y)}{\pi_{old}(y)}R(y)\right].$$

取旧分布 `[0.8,0.2]`、新分布 `[0.2,0.8]`、奖励 `[0,1]`。
旧样本不校正的总体均值是 0.2，目标均值是 0.8。重要性比是 `[0.25,4]`，加权后恢复 0.8。

这是总体恒等式；有限采样仍可能有很大方差。旧概率为 0 的动作，不能凭除法从从未观察到的支持集外恢复出来。裁剪比值也会改变这个等式，不再是原始无偏估计。

In [9]:
# %% 05 重要性比与裁剪目标

def probability_ratio(new_logp: torch.Tensor, old_logp: torch.Tensor) -> torch.Tensor:
    """分母是采样时旧策略；不重估、不传播梯度。拒绝极端比值而非静默裁剪。"""
    require(new_logp.shape == old_logp.shape and new_logp.numel() > 0, "概率形状错误/空批次")
    require(new_logp.device == old_logp.device and new_logp.dtype == old_logp.dtype,
            "概率dtype/device必须一致")
    require(new_logp.is_floating_point() and old_logp.is_floating_point(), "需要浮点log概率")
    require(bool(torch.isfinite(new_logp).all()) and bool(torch.isfinite(old_logp).all()),
            "log概率非有限；不要用clamp隐藏错误")
    log_ratio = new_logp - old_logp.detach()
    require(bool((log_ratio.detach().abs() <= 80).all()), "log比值过大，停止并重新采样/诊断")
    ratio = log_ratio.exp()
    require(bool(torch.isfinite(ratio).all()), "概率比溢出")
    return ratio


def clipped_surrogate(new_logp: torch.Tensor, old_logp: torch.Tensor,
                      advantages: torch.Tensor, epsilon: float = 0.2,
                      clipped: bool = True) -> tuple[torch.Tensor, dict]:
    """对完整回答动作做序列级surrogate；这里的min是最大化目标，loss另取负号。

    固定LOO优势、完整序列ratio；不是逐token PPO，也没有critic/GAE/value loss。
    梯度裁剪在optimizer前另行执行，与此处的目标裁剪不是一回事。
    """
    require(type(epsilon) in (int,float) and math.isfinite(epsilon) and 0 < epsilon < 1,
            "epsilon需要在(0,1)")
    require(type(clipped) is bool, "clipped必须bool")
    require(advantages.shape == new_logp.shape and advantages.device == new_logp.device,
            "优势形状/device错误")
    require(advantages.dtype == new_logp.dtype and bool(torch.isfinite(advantages).all()),
            "优势dtype/数值错误")
    ratio = probability_ratio(new_logp, old_logp)
    advantage = advantages.detach()
    plain = ratio * advantage
    bounded = ratio.clamp(1-epsilon, 1+epsilon) * advantage
    objective = torch.minimum(plain, bounded) if clipped else plain
    outside = (ratio < 1-epsilon) | (ratio > 1+epsilon)
    active_clip = ((ratio > 1+epsilon) & (advantage > 0)) | ((ratio < 1-epsilon) & (advantage < 0))
    loss = -objective.mean()
    return loss, {"ratio":ratio, "objective":objective, "plain":plain,
                  "outside":outside, "active_clip":active_clip,
                  "advantages":advantage}


def sequence_stats(logits: torch.Tensor, labels: torch.Tensor, codec: CharCodec) -> dict:
    """记录各回答前缀上的完整11类分布。提示/PAD不参与统计，EOS参与。"""
    sequence, counts = completion_logps(logits,labels,codec)
    distribution = answer_log_probs(logits[:,:-1,:],codec)
    mask = labels[:,1:].ne(IGNORE)
    return {"sequence":sequence, "distribution":distribution, "mask":mask, "counts":counts}


def old_state_kl(old_distribution: torch.Tensor, new_distribution: torch.Tensor,
                 mask: torch.Tensor) -> torch.Tensor:
    """每个旧轨迹访问前缀的exact categorical KL(old||new)，再对有效位置平均。
    这是已采样状态上的统计，不是遍历所有回答的全序列精确KL。
    """
    require(old_distribution.shape == new_distribution.shape, "分布形状不符")
    require(mask.dtype == torch.bool and mask.shape == old_distribution.shape[:-1]
            and bool(mask.any()), "KL掩码非法")
    require(bool(torch.isfinite(old_distribution).all()) and bool(torch.isfinite(new_distribution).all()),
            "KL分布含非法值")
    for d in (old_distribution,new_distribution):
        require(torch.allclose(d.logsumexp(-1),torch.zeros_like(d[...,0]),atol=2e-5,rtol=2e-5),
                "KL需要归一化后的log分布")
    pointwise = (old_distribution.detach().exp() *
                 (old_distribution.detach()-new_distribution)).sum(-1)
    return pointwise[mask].mean()


def importance_expectation(old: torch.Tensor, current: torch.Tensor,
                           rewards: torch.Tensor) -> dict:
    require(old.ndim == 1 and old.shape == current.shape == rewards.shape, "有限分布形状错误")
    require(all(bool(torch.isfinite(t).all()) for t in (old,current,rewards)), "非有限分布")
    require(bool((old>=0).all()) and bool((current>=0).all()), "概率不能为负")
    require(abs(float(old.sum())-1)<1e-10 and abs(float(current.sum())-1)<1e-10,
            "概率和需要为1")
    require(not bool(((old==0)&(current>0)).any()), "目标分布在旧分布之外有质量，无法重要性校正")
    support=old>0
    ratio=current[support]/old[support]
    return {"old_expectation":float((old*rewards).sum()),
            "current_expectation":float((current*rewards).sum()),
            "weighted_old_expectation":float((old[support]*ratio*rewards[support]).sum()),
            "ratios":ratio.tolist()}


def mechanism_report() -> dict:
    dtype=torch.float64
    importance=importance_expectation(torch.tensor([.8,.2],dtype=dtype),
                                      torch.tensor([.2,.8],dtype=dtype),
                                      torch.tensor([0.,1.],dtype=dtype))
    rows=[]
    for advantage in (1.,-1.):
        for r in (.6,1.,1.5):
            x=torch.tensor([math.log(r)],dtype=dtype,requires_grad=True)
            loss,stats=clipped_surrogate(x,torch.zeros_like(x),torch.full_like(x,advantage),.2)
            derivative=float(torch.autograd.grad(loss,x)[0])
            rows.append({"advantage":advantage,"ratio":r,"objective":float(stats['objective'].detach()),
                         "loss_grad_wrt_new_logp":derivative,
                         "active_clip":bool(stats['active_clip'][0])})
    # 大学习率跨过阈值：目标只去掉局部激励，不做参数投影。
    z=torch.tensor([0.,0.],dtype=dtype,requires_grad=True)
    frozen=F.log_softmax(z.detach().clone(),-1)[0:1]
    loss,_=clipped_surrogate(F.log_softmax(z,-1)[0:1],frozen,torch.ones(1,dtype=dtype))
    grad=torch.autograd.grad(loss,z)[0]
    after=z.detach()-4.0*grad
    ratio_after=float((F.log_softmax(after,-1)[0]-frozen[0]).exp())
    # 一个局部项已经flat，共享logits的其他项仍能推动其概率。
    old=torch.tensor([.3,.4,.3],dtype=dtype)
    current=torch.tensor([.6,.2,.2],dtype=dtype)
    coupled=current.log().requires_grad_()
    loss,info=clipped_surrogate(F.log_softmax(coupled,-1)[:2],old.log()[:2],
                                torch.tensor([1.,1.],dtype=dtype))
    gc=torch.autograd.grad(loss,coupled)[0]
    updated=F.softmax(coupled.detach()-.5*gc,-1)
    # 快照：detach不是独立拷贝。
    source=torch.tensor([1.,2.]); alias=source.detach(); safe=source.detach().clone()
    with torch.no_grad(): source.add_(3)
    # 三类策略的分母区别。
    ref=torch.tensor([.5,.5],dtype=dtype)
    behavior=torch.tensor([.8,.2],dtype=dtype)
    new=torch.tensor([.9,.1],dtype=dtype)
    # 完整轨迹比 = 条件token比的积，不是均值。
    token_ratio=torch.tensor([1.1,1.1,1.1],dtype=dtype)
    # 旧分布下 log(old/new) 单次可能负，不能误称非负精确KL。
    exact_old_new=float((behavior*(behavior.log()-new.log())).sum())
    support_demo=importance_expectation(torch.tensor([.999,.001],dtype=dtype),
                                      torch.tensor([.5,.5],dtype=dtype),
                                      torch.tensor([0.,1.],dtype=dtype))
    return {"importance":importance,"clipping_cases":rows,
            "overshoot":{"epsilon":.2,"sgd_lr":4.,"ratio_after_one_step":ratio_after,
                         "not_a_hard_constraint":ratio_after>1.2},
            "coupling":{"flat_term_ratio_before":2.,"flat_term_probability_before":.6,
                        "flat_term_probability_after":float(updated[0]),
                        "gradient_of_shared_logits":gc.tolist()},
            "snapshots":{"detached_alias":alias.tolist(),"cloned_snapshot":safe.tolist()},
            "three_policies":{"correct_new_over_old":(new/behavior).tolist(),
                              "wrong_new_over_reference":(new/ref).tolist()},
            "sequence_vs_token":{"local_ratios":token_ratio.tolist(),
                                 "sequence_ratio":float(token_ratio.prod()),
                                 "mean_token_ratio":float(token_ratio.mean()),
                                 "long_100_token_ratio_at_1_01":1.01**100},
            "kl":{"old_to_new_exact":exact_old_new,
                  "single_sample_old_minus_new":float(behavior.log()[0]-new.log()[0])},
            "rare_action":support_demo,
            "limits":["mathematical_examples_not_LLM_results","clip_is_not_gradient_clipping",
                       "sequence_surrogate_not_tokenwise_PPO","no_critic_GAE_value_loss"]}

In [10]:
math_report = mechanism_report()
print("精确枚举:", math_report["importance"])
print("旧概率很小的动作:", math_report["rare_action"])
try:
    importance_expectation(torch.tensor([1.,0.],dtype=torch.float64),
                           torch.tensor([.5,.5],dtype=torch.float64),
                           torch.tensor([0.,1.],dtype=torch.float64))
except LessonError as error:
    print("预期拒绝:", error)

精确枚举: {'old_expectation': 0.2, 'current_expectation': 0.8, 'weighted_old_expectation': 0.8, 'ratios': [0.25, 4.0]}
旧概率很小的动作: {'old_expectation': 0.001, 'current_expectation': 0.5, 'weighted_old_expectation': 0.5, 'ratios': [0.5005005005005005, 500.0]}
预期拒绝: 目标分布在旧分布之外有质量，无法重要性校正


## 6. 旧概率必须真的“旧”：detach 不等于快照

同一批回答复用期间，旧 log 概率和优势都固定。不能每次用当前模型刷新分母，否则 ratio 总回到 1，失去旧策略基准。

`tensor.detach()` 切断计算图，但仍共享底层存储。需要独立数据时使用 `detach().clone()`。[PyTorch detach 文档](https://docs.pytorch.org/docs/2.10/generated/torch.Tensor.detach.html)

初始 current=old 时 ratio 的**数值**为 1，分母冻结而分子有梯度，所以梯度不为零；把同一个带图 logp 同时放进分子分母则会抵消。

In [11]:
print("修改原张量后:", math_report["snapshots"])
x = torch.tensor([-0.7,-1.2],dtype=torch.float64,requires_grad=True)
fixed_old = x.detach().clone()
fixed_advantage = torch.tensor([1.,-1.],dtype=torch.float64)
l, info = clipped_surrogate(x,fixed_old,fixed_advantage)
print("初始ratio:", info["ratio"].detach().tolist())
print("loss:", l.item(), "梯度:", torch.autograd.grad(l,x)[0].tolist())

修改原张量后: {'detached_alias': [4.0, 5.0], 'cloned_snapshot': [1.0, 2.0]}
初始ratio: [1.0, 1.0]
loss: -0.0 梯度: [-0.5, 0.5]


## 7. PPO-Clip 公式：先理解 min，再写代码

记 $r=\exp(\log\pi_\theta-\log\pi_{old})$，固定优势为 $A$：

$$L^{clip}=\min\big(rA,\operatorname{clip}(r,1-\epsilon,1+\epsilon)A\big).$$

这是要**最大化**的目标，所以传给梯度下降优化器的是负均值。

- $A>0$：已经提高得过多（$r>1+\epsilon$）时，该项不再获得额外收益。
- $A<0$：已经压低得过多（$r<1-\epsilon$）时，该项不再获得额外收益。

反方向变坏时不能简单切断纠正梯度。`clip(r)*A` **缺少与原项比较的 min，通常不是同一个目标**。[官方公式说明](https://spinningup.openai.com/en/latest/algorithms/ppo.html)

设 epsilon=0.2，先手算下面六行。

In [12]:
print("advantage  ratio  objective  d(loss)/d(new_logp)  active_clip")
for row in math_report["clipping_cases"]:
    print(f"{row['advantage']:9.1f} {row['ratio']:6.2f} {row['objective']:10.3f} "
          f"{row['loss_grad_wrt_new_logp']:19.3f} {row['active_clip']}")

advantage  ratio  objective  d(loss)/d(new_logp)  active_clip
      1.0   0.60      0.600              -0.600 False
      1.0   1.00      1.000              -1.000 False
      1.0   1.50      1.200               0.000 True
     -1.0   0.60     -0.800               0.000 True
     -1.0   1.00     -1.000               1.000 False
     -1.0   1.50     -1.500               1.500 False


## 8. 目标裁剪，不是硬信赖域，也不是梯度裁剪

一次较大的优化步就可能越过 `[0.8,1.2]`。后续某个样本项进入平坦区域，也不能阻止共享参数被其他样本或优化器动量继续改变。

本课分别验证：

1. 初始概率比为 1，一次较大 SGD 后超过 1.2。
2. 一个样本自己的项已平坦，但其他样本仍改变它的概率。

`clip_grad_norm_` 限制的是参数梯度范数，是另一个独立操作。
官方 PPO 说明也明确指出裁剪不能保证小 KL，因此实现可能另加 KL early stop。[PPO 说明](https://spinningup.openai.com/en/latest/algorithms/ppo.html)

In [13]:
print("大步更新:", math_report["overshoot"])
print("共享参数联动:", math_report["coupling"])

大步更新: {'epsilon': 0.2, 'sgd_lr': 4.0, 'ratio_after_one_step': 1.9640275800758167, 'not_a_hard_constraint': True}
共享参数联动: {'flat_term_ratio_before': 2.0, 'flat_term_probability_before': 0.6, 'flat_term_probability_after': 0.5722441760120609, 'gradient_of_shared_logits': [0.14999999999999994, -0.19999999999999996, 0.04999999999999998]}


## 9. 序列比与逐 token 比：本课选的是哪一个？

完整回答的比值：

$$r_{seq}=\exp\sum_t(\log\pi_\theta(y_t|x,y_{<t})-\log\pi_{old}(y_t|x,y_{<t}))
=\prod_t r_t.$$

三个 token 比值均为 1.1，则完整回答比为 **1.331**。分别裁剪每个 token，再求和，与裁剪整个序列比，是不同目标。

本课为了衔接上一课，把**整段回答看成上下文 bandit 的一个动作**，使用序列级裁剪＋固定 LOO 优势。原始 PPO 通常按时间步动作、状态和优势构造代理目标，并含 value function 等部分。本课没有 critic/GAE，不能称为完整逐 token PPO 或 GRPO。

长回答的概率比会连乘，方差与数值问题更加明显。本课只使用三步回答；不会把这种实现未经修改推广到长推理链。

In [14]:
print(json.dumps(math_report["sequence_vs_token"],ensure_ascii=False,indent=2))

{
  "local_ratios": [
    1.1,
    1.1,
    1.1
  ],
  "sequence_ratio": 1.3310000000000004,
  "mean_token_ratio": 1.1,
  "long_100_token_ratio_at_1_01": 2.7048138294215285
}


## 10. KL 诊断：方向、采样分布与范围必须标明

本课保存每个旧回答前缀上的 11 类行为分布，计算该前缀的精确 categorical KL：

$$D_{KL}(\pi_{old}(\cdot|s)\|\pi_\theta(\cdot|s)).$$

然后只在旧轨迹访问的有效回答位置上平均，记为 `old_state_kl`。这不是遍历所有可能前缀后的全局 KL，也不是完整序列分布的精确 KL。

另一个字段 `current_reference_kl_on_old_states` 只监测对固定 SFT 参考的漂移，不参与 loss。单条序列的 `logp_old-logp_current` 可为负，因此不会冒称它是非负的精确 KL。

存整张旧分布只因本课动作数是 11。真实大词表需要重新设计日志/估计方案，不能无脑保存所有位置所有概率。

In [15]:
p_old = torch.tensor([[[.8,.2],[.5,.5]]],dtype=torch.float64).log()
p_new = torch.tensor([[[.9,.1],[.01,.99]]],dtype=torch.float64).log()
valid = torch.tensor([[True,False]])
print("有效前缀上的精确KL:", old_state_kl(p_old,p_new,valid).item())
print("单个动作的old-new log比:", math_report["kl"]["single_sample_old_minus_new"])

有效前缀上的精确KL: 0.0444030075868823
单个动作的old-new log比: -0.11778303565638343


## 11. 有限复用循环：哪些量不能变化？

每批先在当前策略下独立采样同题四次，得到奖励与 LOO 优势，再封存输入、旧概率、旧分布、参考诊断分布和优势。

```text
新采样批次：ratio=1
   ↓
最多4次局部更新：旧概率和优势不动，当前概率每步重算
   ↓
达到上限或KL停止：关闭批次
   ↓
重新采样，建立新的行为旧策略基准
```

批次 ID、内容摘要和更新后策略摘要用于发现意外修改、跨批次混用和控制器之外的参数更新，**不是针对恶意代码的安全认证**。

`clip_stop` 在下一次更新前检查阈值，超出就不再执行；不回滚刚才造成超限的那一步，也不宣称提供硬 KL 保证。

In [16]:
# %% 06 封存旧策略信息，有限次复用同一批次

def batch_seal(batch: dict) -> str:
    tensor_part=tensor_hash({k:batch[k] for k in (
        "input_ids","labels","old_logps","old_distribution","reference_distribution",
        "advantages","rewards","mask")})
    metadata={k:batch[k] for k in ("rows","answers","samples","behavior_hash","codec_hash",
                                  "batch_id","group_size","max_reuses")}
    return digest({"tensors":tensor_part,"metadata":metadata})


def prepare_reuse_batch(policy: TinyLM, reference: TinyLM, rows: list[dict],
                        codec: CharCodec, cfg: TrainConfig, generator: torch.Generator,
                        batch_id: int) -> dict:
    cfg.validate()
    require(all(r['split']=='train' for r in rows), "复用批次只能来自训练侧")
    require(type(batch_id) is int and batch_id>=0,"batch_id非法")
    require(all(not p.requires_grad for p in reference.parameters()),"参考必须冻结")
    rollout=sample_rollouts(policy,rows,codec,cfg.group_size,generator,3)
    inputs=collate_completions(rollout['rows'],rollout['answers'],codec,policy.cfg.max_len,
                              next(policy.parameters()).device)
    with torch.no_grad(),eval_mode(policy),eval_mode(reference):
        old=sequence_stats(policy(inputs['input_ids']),inputs['labels'],codec)
        ref=sequence_stats(reference(inputs['input_ids']),inputs['labels'],codec)
    sampled=torch.tensor([s['behavior_logp'] for s in rollout['samples']],
                         device=old['sequence'].device,dtype=old['sequence'].dtype)
    error=float((old['sequence']-sampled).abs().max())
    require(error<3e-5,"采样与重算概率不一致：检查温度、动作集合、移位、Dropout")
    rewards=old['sequence'].new_tensor([s['reward'] for s in rollout['samples']])
    advantages=leave_one_out(rewards.reshape(len(rows),cfg.group_size)).reshape(-1)
    batch={**{k:t.detach().clone() for k,t in inputs.items()},
           "old_logps":old['sequence'].detach().clone(),
           "old_distribution":old['distribution'].detach().clone(),
           "reference_distribution":ref['distribution'].detach().clone(),
           "mask":old['mask'].detach().clone(),"advantages":advantages.detach().clone(),
           "rewards":rewards.detach().clone(),"rows":rollout['rows'],"answers":rollout['answers'],
           "samples":rollout['samples'],"behavior_hash":rollout['policy_hash'],
           "codec_hash":rollout['codec_hash'],"batch_id":batch_id,"group_size":cfg.group_size,
           "max_reuses":cfg.inner_epochs,"updates":0,"closed":False,
           "expected_policy_hash":rollout['policy_hash'],"initial_recompute_error":error}
    batch['seal']=batch_seal(batch)
    return batch


def validate_reuse_batch(policy: TinyLM,batch: dict,codec: CharCodec,
                         active_batch_id: int) -> None:
    require(not batch['closed'],"批次已结束，需要重新采样")
    require(batch['batch_id']==active_batch_id,"不能把旧rollout混入下一批")
    require(batch['updates'] < batch['max_reuses'],"超过本批最大复用次数")
    require(batch['codec_hash']==digest(codec.contract()),"采样契约改变")
    require(batch['seal']==batch_seal(batch),"旧概率/优势/输入/元数据被修改")
    require(batch['expected_policy_hash']==tensor_hash(policy.state_dict()),
            "策略被控制循环之外的更新改动，拒绝不明轨迹")
    require(all(r['split']=='train' for r in batch['rows']),"测试数据不能进入更新")
    require(all(not batch[k].requires_grad for k in ("old_logps","old_distribution","advantages")),
            "封存量不能持有梯度图")


@torch.no_grad()
def batch_diagnostics(policy: TinyLM,batch: dict,codec: CharCodec,epsilon: float) -> dict:
    with eval_mode(policy):
        current=sequence_stats(policy(batch['input_ids']),batch['labels'],codec)
    ratio=probability_ratio(current['sequence'],batch['old_logps'])
    a=batch['advantages']
    active=((ratio>1+epsilon)&(a>0))|((ratio<1-epsilon)&(a<0))
    point_ref=(current['distribution'].exp()*(current['distribution']-batch['reference_distribution'])).sum(-1)
    return {"ratio_min":float(ratio.min()),"ratio_max":float(ratio.max()),
            "ratio_mean":float(ratio.mean()),
            "ratio_outside_fraction":float(((ratio<1-epsilon)|(ratio>1+epsilon)).float().mean()),
            "active_clip_fraction":float(active.float().mean()),
            "old_state_kl":max(0.,float(old_state_kl(batch['old_distribution'],current['distribution'],batch['mask']))),
            "current_reference_kl_on_old_states":max(0.,float(point_ref[batch['mask']].mean())),
            "sampled_sequence_old_minus_current":float((batch['old_logps']-current['sequence']).mean()),
            "note":"KL在旧样本访问的回答前缀上平均；不是全状态或完整序列精确KL"}


def reuse_step(policy: TinyLM,batch: dict,codec: CharCodec,optimizer,
               cfg: TrainConfig,mode: str,active_batch_id: int) -> dict:
    require(mode in {'is','clip','clip_stop'},"未知更新模式")
    validate_reuse_batch(policy,batch,codec,active_batch_id)
    trainable=[p for p in policy.parameters() if p.requires_grad]
    require({id(p) for group in optimizer.param_groups for p in group['params']}=={id(p) for p in trainable},
            "优化器必须恰好包含当前LoRA参数")
    before=batch_diagnostics(policy,batch,codec,cfg.clip_epsilon)
    # 这是停止后续步，不回滚已经提交的上一更新，也不是硬KL约束。
    if mode=='clip_stop' and before['old_state_kl']>cfg.target_state_kl:
        batch['closed']=True
        return {"updated":False,"reason":"old_state_kl_stop","before":before,
                "updates_so_far":batch['updates']}
    optimizer.zero_grad(set_to_none=True)
    with eval_mode(policy):
        now,_=completion_logps(policy(batch['input_ids']),batch['labels'],codec)
        loss,_=clipped_surrogate(now,batch['old_logps'],batch['advantages'],cfg.clip_epsilon,
                                clipped=(mode!='is'))
        loss.backward()
    for name,p in policy.named_parameters():
        if p.requires_grad:
            require(p.grad is not None and bool(torch.isfinite(p.grad).all()),"adapter梯度缺失或非有限")
        else:
            require(p.grad is None,"冻结基座不应获得参数梯度")
    norm=nn.utils.clip_grad_norm_(trainable,cfg.max_grad_norm,error_if_nonfinite=True)
    optimizer.step()
    batch['updates']+=1
    batch['expected_policy_hash']=tensor_hash(policy.state_dict())
    require(batch['seal']==batch_seal(batch),"更新后旧概率/优势必须保持不变")
    after=batch_diagnostics(policy,batch,codec,cfg.clip_epsilon)
    if batch['updates']>=batch['max_reuses']:
        batch['closed']=True
    return {"updated":True,"inner_update":batch['updates'],"surrogate_loss":float(loss.detach()),
            "gradient_norm_before_clip":float(norm),"before":before,"after":after,
            "reward_of_fixed_batch":float(batch['rewards'].mean())}

In [17]:
probe_cfg = TrainConfig(sft_steps=2,rollout_batches=2,inner_epochs=2)
probe_batch = prepare_reuse_batch(
    probe_policy,probe_reference,splits["train"][:2],codec,probe_cfg,
    torch.Generator().manual_seed(600),batch_id=0)
print("样本数量:", len(probe_batch["answers"]))
print("采样与重算最大差异:", probe_batch["initial_recompute_error"])
print("初始诊断:", batch_diagnostics(probe_policy,probe_batch,codec,.2))
probe_opt = torch.optim.Adam([p for p in probe_policy.parameters() if p.requires_grad],lr=.003)
old_snapshot = probe_batch["old_logps"].clone()
for _ in range(2):
    item = reuse_step(probe_policy,probe_batch,codec,probe_opt,probe_cfg,"clip",0)
    print("更新:",item["inner_update"],"surrogate loss:",item["surrogate_loss"])
print("旧概率未变:", torch.equal(old_snapshot,probe_batch["old_logps"]))
print("批次关闭:", probe_batch["closed"])
# 未预热的策略很可能整组奖励0；零loss并不代表它已学会任务。

样本数量: 8
采样与重算最大差异: 4.76837158203125e-07
初始诊断: {'ratio_min': 1.0, 'ratio_max': 1.0, 'ratio_mean': 1.0, 'ratio_outside_fraction': 0.0, 'active_clip_fraction': 0.0, 'old_state_kl': 0.0, 'current_reference_kl_on_old_states': 0.0, 'sampled_sequence_old_minus_current': 0.0, 'note': 'KL在旧样本访问的回答前缀上平均；不是全状态或完整序列精确KL'}


更新: 1 surrogate loss: -0.0
更新: 2 surrogate loss: -0.0
旧概率未变: True
批次关闭: True


## 12. 完整训练对照：固定配置，不根据测试结果调参数

三路从同一份 SFT 权重和 LoRA 初始化开始：

| 路径 | 同批最多更新 | 比值裁剪 | KL停止 |
|---|---:|---|---|
| `is` | 4 | 无 | 无 |
| `clip` | 4 | epsilon=0.2 | 无 |
| `clip_stop` | 4 | epsilon=0.2 | `old_state_kl > 0.02` 后停止后续步 |

固定 24 批新轨迹、每批 8 问题×4回答，各路采样 768 条回答。最多 96 次优化器更新。共享问题调度与初始种子，但策略变化后回答和长度可能不同；早停还会改变实际更新次数，**不是等GPU成本对照**。

参考模型始终冻结，只做诊断，不添加参考KL惩罚。开发集只观察，不挑选轮次；三个分支保存固定最后一步后，才首次读取测试集。

这次采用明文数学定义的三个actor分支，不训练价值网络，不复现完整PPO基准。

In [18]:
# %% 07 训练、保存与留出评测；测试不参与超参数/步数选择

def run_experiment(output_root=Path('lesson_60_outputs'), cfg: TrainConfig|None=None,
                   device_name='cpu',mode='all') -> Path:
    cfg=cfg or TrainConfig();cfg.validate();device=resolve_device(device_name)
    require(mode in {'all','is','clip','clip_stop'},'mode非法')
    out=Path(output_root)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
    out.mkdir(parents=True,exist_ok=False)
    manifest={"version":VERSION,"python":platform.python_version(),"torch":str(torch.__version__),
              "device":str(device),"status":"running","config":asdict(cfg),"sources":SOURCES,
              "boundaries":["sequence_level_clipped_actor_surrogate","fixed_reward_LOO_not_critic_GAE",
                "reference_for_diagnostics_only_no_reference_KL_penalty","no_test_selection",
                "single_seed_synthetic_task","not_pretrained_LLM","no_exact_training_resume"]}
    write_json(out/'manifest.json',manifest)
    try:
        seed_all(cfg.seed)
        codec=CharCodec();splits=make_splits(cfg.seed);validate_splits(splits)
        model_cfg=ModelConfig(vocab_size=len(codec.tokens));base=TinyLM(model_cfg).to(device)
        print('SFT预热...',flush=True)
        warmup=sft_warmup(base,splits['train'],codec,cfg)
        reference=freeze_reference(base);ref_hash=tensor_hash(reference.state_dict())
        template=inject_lora(copy.deepcopy(base),cfg.rank,cfg.alpha).eval()
        schedule_rng=random.Random(cfg.seed+40)
        schedule=[schedule_rng.sample(splits['train'],cfg.prompts_per_update)
                  for _ in range(cfg.rollout_batches)]
        write_json(out/'split.json',splits)
        write_json(out/'schedule.json',[[r['id'] for r in group] for group in schedule])
        write_json(out/'mechanisms.json',mechanism_report())
        torch.save({"version":VERSION,"config":asdict(model_cfg),"codec":codec.contract(),
                    "state":{n:t.detach().cpu().clone() for n,t in reference.state_dict().items()},
                    "reference_hash":ref_hash},out/'reference.pt')
        summary={"model_parameters":sum(p.numel() for p in reference.parameters()),
                 "adapter_parameters":sum(p.numel() for p in template.parameters() if p.requires_grad),
                 "sft_history":warmup,"reference_hash":ref_hash,"arms":{},
                 "selection":"fixed_last_rollout_no_test_selection"}
        trained={}
        for name in (['is','clip','clip_stop'] if mode=='all' else [mode]):
            policy=copy.deepcopy(template).to(device)
            optimizer=torch.optim.Adam([p for p in policy.parameters() if p.requires_grad],lr=cfg.rl_lr)
            generator=torch.Generator(device=device).manual_seed(cfg.seed+100)
            history=[];dev=[];updates=0;completions=0;generated_tokens=0;stopped=0
            print('训练分支:',name,flush=True)
            with (out/(name+'_rollouts.jsonl')).open('w',encoding='utf-8') as stream:
                for bid,rows in enumerate(schedule):
                    batch=prepare_reuse_batch(policy,reference,rows,codec,cfg,generator,bid)
                    initial=batch_diagnostics(policy,batch,codec,cfg.clip_epsilon)
                    require(abs(initial['ratio_mean']-1)<3e-5,'新批次初始ratio必须为1')
                    record={"batch_id":bid,"initial":initial,"inner":[],
                            "sampled_reward":float(batch['rewards'].mean()),
                            "zero_advantage_fraction":float((batch['advantages']==0).float().mean())}
                    while not batch['closed']:
                        step=reuse_step(policy,batch,codec,optimizer,cfg,name,bid)
                        record['inner'].append(step)
                        updates+=int(step['updated'])
                        stopped+=int(not step['updated'])
                    require(batch['seal']==batch_seal(batch),'封存数据被修改')
                    stream.write(canonical({"batch_id":bid,"behavior_hash":batch['behavior_hash'],
                                            "samples":batch['samples'],
                                            "advantages":batch['advantages'].cpu().tolist(),
                                            "old_logps":batch['old_logps'].cpu().tolist(),
                                            "reuse_updates":batch['updates']})+'\n')
                    completions+=len(batch['answers']);generated_tokens+=sum(map(len,batch['answers']))
                    history.append(record)
                    if bid==0 or (bid+1)%8==0 or bid+1==cfg.rollout_batches:
                        d=evaluate_policy(policy,splits['dev'],codec,4,seed=9060)
                        dev.append({"rollout_batch":bid+1,**compact_eval(d)})
                        print(f'  batch {bid+1}: updates={updates}, dev P(correct)={d["exact_success_probability"]:.4f}',flush=True)
            require(tensor_hash(base_state(policy))==ref_hash,'冻结基座被更新')
            require(tensor_hash(reference.state_dict())==ref_hash,'固定参考被更新')
            require(all(p.grad is None for p in reference.parameters()),'参考收到梯度')
            torch.save({"version":VERSION,"arm":name,"train_config":asdict(cfg),
                        "reference_hash":ref_hash,"codec":codec.contract(),
                        "adapter":adapter_snapshot(policy)},out/(name+'.pt'))
            summary['arms'][name]={"updates":updates,"rollout_batches":len(history),
                  "completions":completions,"generated_tokens":generated_tokens,
                  "stopped_batches":stopped,"history":history,"dev_history":dev,
                  "reference_unchanged":True,"base_unchanged":True,
                  "policy_hash":tensor_hash(policy.state_dict())}
            trained[name]=policy.cpu()
        # 所有分支都已保存固定最后一步；现在才首次读取test作最终比较。
        evaluation={"reference":evaluate_policy(reference,splits['test'],codec,cfg.eval_samples,seed=906060)}
        for name,policy in trained.items():
            policy=policy.to(device)
            evaluation[name]=evaluate_policy(policy,splits['test'],codec,cfg.eval_samples,seed=906060)
            summary['arms'][name]['test']=compact_eval(evaluation[name])
        summary['reference_test']=compact_eval(evaluation['reference'])
        write_json(out/'training.json',summary);write_json(out/'evaluation.json',evaluation)
        reload_report=reload_run(out,device_name);write_json(out/'reload.json',reload_report)
        manifest['status']='completed';write_json(out/'manifest.json',manifest)
        print('输出目录:',out,flush=True)
        return out
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        write_json(out/'manifest.json',manifest)
        raise


def reload_run(directory: Path,device_name='cpu') -> dict:
    directory=Path(directory);device=resolve_device(device_name)
    blob=torch.load(directory/'reference.pt',map_location='cpu',weights_only=True)
    require(blob['version']==VERSION,'检查点版本错误')
    codec=CharCodec();require(blob['codec']==codec.contract(),'动作/token契约改变')
    base=TinyLM(ModelConfig(**blob['config'])).to(device)
    base.load_state_dict(blob['state'],strict=True);base.eval().requires_grad_(False)
    require(tensor_hash(base.state_dict())==blob['reference_hash'],'参考文件内容错误')
    evaluation=json.loads((directory/'evaluation.json').read_text(encoding='utf-8'))
    splits=json.loads((directory/'split.json').read_text(encoding='utf-8'));validate_splits(splits)
    result={}
    for name in sorted(set(evaluation)-{'reference'}):
        ck=torch.load(directory/(name+'.pt'),map_location='cpu',weights_only=True)
        require(ck['version']==VERSION and ck['reference_hash']==blob['reference_hash'],'adapter接错参考')
        require(ck['codec']==codec.contract(),'adapter契约不符')
        cfg=TrainConfig(**ck['train_config']);cfg.validate()
        policy=inject_lora(copy.deepcopy(base),cfg.rank,cfg.alpha)
        load_adapter(policy,ck['adapter']);policy.eval().requires_grad_(False)
        report=evaluate_policy(policy,splits['test'],codec,cfg.eval_samples,seed=906060)
        old=evaluation[name]
        err=max(abs(a['p']-b['p']) for a,b in zip(report['per_task_probability'],old['per_task_probability']))
        equal=[r['token_ids'] for r in report['greedy']]==[r['token_ids'] for r in old['greedy']]
        require(err<2e-5 and equal,'重载输出不一致；跨设备不承诺逐位相同')
        result[name]={"max_probability_error":err,"greedy_ids_equal":equal,
                      "trainable_parameters":sum(p.numel() for p in policy.parameters() if p.requires_grad)}
    return {"arms":result,"scope":"inference_reload_not_exact_training_resume"}

In [19]:
RUN_DIRECTORY = None
if RUN_TRAINING:
    RUN_DIRECTORY = run_experiment(OUTPUT_ROOT,TRAIN_CONFIG,DEVICE,"all")
else:
    print("已明确跳过完整训练；没有填入模拟训练成绩。")

SFT预热...


训练分支: is


  batch 1: updates=4, dev P(correct)=0.2518


  batch 8: updates=32, dev P(correct)=0.2555


  batch 16: updates=64, dev P(correct)=0.2546


  batch 24: updates=96, dev P(correct)=0.2616


... (7 outputs left out)

  batch 8: updates=32, dev P(correct)=0.2557


  batch 16: updates=64, dev P(correct)=0.2572


  batch 24: updates=96, dev P(correct)=0.2584


输出目录: lesson_60_outputs/run_20261103_031240_552206


## 13. 同时看采样概率、实际样本和贪心结果

本任务只有一条可接受的完整轨迹（两位数字＋EOS），因此能精确计算它的概率乘积。它是本课受限三步策略的成功概率，不是遍历一般开放式任务的正确率。

独立评测还执行每题8次采样和一次贪心。三项指标分别回答不同的问题。下面读取本次真实训练输出，不填入预设成绩。

In [20]:
if RUN_DIRECTORY is not None:
    training_report = json.loads((RUN_DIRECTORY/"training.json").read_text(encoding="utf-8"))
    print("模式              P(正确)     采样成功     贪心成功   优化器更新")
    base_test = training_report["reference_test"]
    print(f"{'SFT':16} {base_test['exact_success_probability']:.6f} "
          f"{int(base_test['sampled_successes'])}/{base_test['sampled_total']:3d}        "
          f"{int(base_test['greedy_successes'])}/48       --")
    for name,arm in training_report["arms"].items():
        e = arm["test"]
        print(f"{name:16} {e['exact_success_probability']:.6f} "
              f"{int(e['sampled_successes'])}/{e['sampled_total']:3d}        "
              f"{int(e['greedy_successes'])}/48       {arm['updates']}")
else:
    print("没有执行本次训练，因此不显示成绩。")

模式              P(正确)     采样成功     贪心成功   优化器更新
SFT              0.227148 96/384        25/48       --
clip             0.235430 99/384        21/48       96
clip_stop        0.235430 99/384        21/48       96
is               0.239344 105/384        21/48       96


### 解释结果时，不要只挑最好看的数字

在随本讲义提供的 CPU 默认运行中，SFT 贪心为25/48，三个训练分支为21/48；目标轨迹的平均概率有所上升。固定候选概率提高与贪心解码改进不是同一个保证。

`clip_stop` 的默认阈值未触发，因此和 `clip` 输出相同。**没有为了让三行看起来不同，就在看过测试结果后调整阈值。** 数学与固定检查中另有强制超限案例，验证停止路径。

这些是单种子、人工复制任务，不能给PPO、裁剪或早停做普遍排名。

In [21]:
if RUN_DIRECTORY is not None:
    for name,arm in training_report["arms"].items():
        steps = [s for h in arm["history"] for s in h["inner"] if s["updated"]]
        print(name,{
            "完整回答采样":arm["completions"],"实际生成token":arm["generated_tokens"],
            "实际更新":arm["updates"],"KL停止批次数":arm["stopped_batches"],
            "更新后最大序列比":max(s["after"]["ratio_max"] for s in steps),
            "最大旧状态KL":max(s["after"]["old_state_kl"] for s in steps),
            "最大有效裁剪比例":max(s["after"]["active_clip_fraction"] for s in steps),
        })

clip {'完整回答采样': 768, '实际生成token': 2255, '实际更新': 96, 'KL停止批次数': 0, '更新后最大序列比': 1.651910424232483, '最大旧状态KL': 0.010650678537786007, '最大有效裁剪比例': 0.1875}
clip_stop {'完整回答采样': 768, '实际生成token': 2255, '实际更新': 96, 'KL停止批次数': 0, '更新后最大序列比': 1.651910424232483, '最大旧状态KL': 0.010650678537786007, '最大有效裁剪比例': 0.1875}
is {'完整回答采样': 768, '实际生成token': 2256, '实际更新': 96, 'KL停止批次数': 0, '更新后最大序列比': 2.450537919998169, '最大旧状态KL': 0.02400636114180088, '最大有效裁剪比例': 0.25}


## 14. 保存重载与复现边界

`reference.pt` 保存 SFT 结构、词表、参数和身份；每个分支保存自己的 adapter 与配置。`*_rollouts.jsonl` 保存真实采样、奖励、旧log概率、固定优势和复用次数。

重载时重建干净基座，核对来源，再加载adapter。这里保存的是推理状态，不是优化器、rollout游标与RNG齐全的断点续训文件。

相同CPU环境可核对结果；不同设备与PyTorch版本不保证逐位一致。[PyTorch复现说明](https://docs.pytorch.org/docs/2.10/notes/randomness.html)

In [22]:
if RUN_DIRECTORY is not None:
    print(json.dumps(reload_run(RUN_DIRECTORY,DEVICE),ensure_ascii=False,indent=2))
else:
    print("训练已跳过，没有本次检查点可重载。")

{
  "arms": {
    "clip": {
      "max_probability_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    },
    "clip_stop": {
      "max_probability_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    },
    "is": {
      "max_probability_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    }
  },
  "scope": "inference_reload_not_exact_training_resume"
}


## 15. 自动检查：不仅看 loss 能否运行

检查覆盖正负优势、符号、旧概率冻结、支持集、序列与token比的差异、初始梯度、KL方向、旧批次身份、真实模型梯度、参考冻结和磁盘重载。

`ratio_outside_fraction` 统计超出区间的样本；`active_clip_fraction` 还结合优势方向。二者不能当成同一个指标。

另外，零优势不一定让有历史动量的优化器完全不动，因此本课不使用“loss为零”作为参数必定不变的证明。

In [23]:
# %% 08 自动检查：公式、梯度、数据身份与实际训练连接
class Checks:
    def __init__(self): self.names=[]
    def ok(self,name,condition):
        if not bool(condition): raise AssertionError(name)
        self.names.append(name)
    def close(self,name,a,b,atol=1e-10,rtol=1e-10):
        self.ok(name,torch.allclose(torch.as_tensor(a,dtype=torch.float64),
                                   torch.as_tensor(b,dtype=torch.float64),atol=atol,rtol=rtol))
    def raises(self,name,fn):
        try: fn()
        except (LessonError,ValueError): self.names.append(name);return
        raise AssertionError(name+': 应拒绝')


def run_checks() -> dict:
    seed_all(6001);c=Checks();dtype=torch.float64
    mech=mechanism_report()
    c.close('旧分布直接平均与当前目标不同',mech['importance']['old_expectation'],.2)
    c.close('有限分布重要性校正精确',mech['importance']['weighted_old_expectation'],.8)
    c.close('当前分布期望',mech['importance']['current_expectation'],.8)
    c.raises('支持集不覆盖时拒绝重要性校正',lambda:importance_expectation(
        torch.tensor([1.,0.],dtype=dtype),torch.tensor([.5,.5],dtype=dtype),torch.ones(2,dtype=dtype)))
    c.raises('非归一化概率拒绝',lambda:importance_expectation(
        torch.tensor([1.,1.],dtype=dtype),torch.tensor([.5,.5],dtype=dtype),torch.ones(2,dtype=dtype)))
    c.ok('罕见动作的重要性比可以很大',mech['rare_action']['ratios'][1]==500)
    # 在有限单步动作上用完整求和对比精确目标与IS梯度。
    z=torch.tensor([.3,-.2,.1],dtype=dtype,requires_grad=True)
    old=torch.tensor([.2,.5,.3],dtype=dtype);reward=torch.tensor([1.,0.,.4],dtype=dtype)
    new=F.softmax(z,-1)
    truth=torch.autograd.grad((new*reward).sum(),z,retain_graph=True)[0]
    ratios=probability_ratio(F.log_softmax(z,-1),old.log())
    estimate=torch.autograd.grad((old*ratios*reward).sum(),z)[0]
    c.close('全动作重要性目标梯度与精确期望一致',estimate,truth)
    newlog=torch.tensor([math.log(.6),math.log(1.5),math.log(.6),math.log(1.5)],
                        dtype=dtype,requires_grad=True)
    oldlog=torch.zeros(4,dtype=dtype,requires_grad=True)
    advantage=torch.tensor([1.,1.,-1.,-1.],dtype=dtype,requires_grad=True)
    loss,info=clipped_surrogate(newlog,oldlog,advantage,.2)
    g,go,ga=torch.autograd.grad(loss,(newlog,oldlog,advantage),allow_unused=True)
    c.close('四象限目标',[float(v) for v in info['objective'].detach()],[.6,1.2,-.8,-1.5])
    c.close('四象限梯度方向及平坦区域',g,[-.6/4,0.,0.,1.5/4])
    c.ok('old_logp不参与反向',go is None)
    c.ok('固定优势不参与反向',ga is None)
    c.close('标记真正裁剪的两个样本',info['active_clip'],[False,True,True,False])
    c.close('超出区间的四个样本不全等于有效裁剪',info['outside'],[True,True,True,True])
    c.ok('损失是最大化目标的负均值',abs(float(loss.detach())-0.125)<1e-12)
    one=torch.tensor([-.7,-1.2],dtype=dtype,requires_grad=True)
    oldsame=one.detach().clone();a=torch.tensor([1.,-1.],dtype=dtype)
    loss1,stats=clipped_surrogate(one,oldsame,a)
    grad1=torch.autograd.grad(loss1,one)[0]
    c.close('初始概率比为1',stats['ratio'],[1.,1.])
    c.close('初始clip梯度等于REINFORCE代理梯度',grad1,-a/2)
    c.close('优势均值为零初始loss仍可为零',float(loss1.detach()),0.)
    c.ok('零loss不等于零梯度',float(grad1.abs().sum())>0)
    isloss,isinfo=clipped_surrogate(newlog,oldlog,advantage,.2,clipped=False)
    c.close('IS分支不进行surrogate裁剪',isinfo['objective'],[.6,1.5,-.6,-1.5])
    c.ok('裁剪目标不高于未裁剪目标',bool((info['objective']<=info['plain']+1e-12).all()))
    c.raises('非法epsilon拒绝',lambda:clipped_surrogate(one,oldsame,a,1.))
    c.raises('优势形状不符拒绝',lambda:clipped_surrogate(one,oldsame,a[:1]))
    c.raises('优势NaN拒绝',lambda:clipped_surrogate(one,oldsame,torch.full_like(a,float('nan'))))
    c.raises('空概率拒绝',lambda:probability_ratio(torch.empty(0),torch.empty(0)))
    c.raises('非有限概率拒绝',lambda:probability_ratio(torch.tensor([float('inf')]),torch.zeros(1)))
    c.raises('过大比值不静默clamp',lambda:probability_ratio(torch.tensor([81.]),torch.zeros(1)))
    c.ok('一次更新可越过1+epsilon',mech['overshoot']['ratio_after_one_step']>1.9)
    c.ok('局部项已flat也可被共享参数改变',abs(mech['coupling']['flat_term_probability_after']-.6)>.01)
    c.close('detach仍共享存储',mech['snapshots']['detached_alias'],[4.,5.])
    c.close('detach_clone保持旧快照',mech['snapshots']['cloned_snapshot'],[1.,2.])
    c.ok('旧策略不同于参考时分母不同',mech['three_policies']['correct_new_over_old']!=mech['three_policies']['wrong_new_over_reference'])
    c.close('三步序列比是乘积',mech['sequence_vs_token']['sequence_ratio'],1.331)
    c.ok('token均比不是轨迹比',abs(mech['sequence_vs_token']['sequence_ratio']-mech['sequence_vs_token']['mean_token_ratio'])>.2)
    c.ok('各token小偏移会积累',mech['sequence_vs_token']['long_100_token_ratio_at_1_01']>2.7)
    c.ok('单样本old_minus_new可为负',mech['kl']['single_sample_old_minus_new']<0)
    c.ok('同分布全KL非负',mech['kl']['old_to_new_exact']>0)
    olddist=torch.tensor([[[.8,.2],[.5,.5]]],dtype=dtype).log()
    newdist=torch.tensor([[[.9,.1],[.01,.99]]],dtype=dtype).log().requires_grad_()
    mask=torch.tensor([[True,False]])
    c.close('KL只平均有效前缀',old_state_kl(olddist,newdist,mask),mech['kl']['old_to_new_exact'])
    c.close('同分布状态KL为零',old_state_kl(olddist,olddist,mask),0.)
    c.raises('全空KL掩码拒绝',lambda:old_state_kl(olddist,newdist,torch.zeros_like(mask)))
    c.raises('未归一化log分布拒绝',lambda:old_state_kl(olddist+2,newdist,mask))

    codec=CharCodec();splits=make_splits(60);validate_splits(splits)
    c.ok('划分固定160_48_48',[len(splits[k]) for k in ('train','dev','test')]==[160,48,48])
    c.ok('固定11种回答动作',len(codec.answer_ids)==11)
    c.ok('精确回答EOS奖1',score_completion('02','eos','02')['reward']==1)
    c.ok('无EOS不能冒充正确',score_completion('02','max_tokens','02')['reward']==0)
    c.raises('执行故障不能作为零奖励',lambda:score_completion('02','error','02'))
    rows=splits['train'][:2]
    answers=[codec.encode(rows[0]['answer'])+[EOS],[EOS]]
    batch=collate_completions(rows,answers,codec,24)
    x=torch.zeros(2,batch['input_ids'].shape[1],len(codec.tokens),dtype=dtype,requires_grad=True)
    seq,counts=completion_logps(x,batch['labels'],codec)
    c.close('回答求和包括EOS',seq,[-3*math.log(11),-math.log(11)])
    c.close('提示和PAD排除',counts,[3,1])
    grad=torch.autograd.grad(seq.sum(),x)[0];first=1+len(rows[0]['prompt'])-1
    c.ok('正确移位到首个回答的前一位置',float(grad[:,first].abs().sum())>0)
    c.ok('较早提示logits不直接计损失',bool((grad[:,:first]==0).all()))
    c.raises('EOS后不可继续',lambda:collate_completions(rows[:1],[[EOS,EOS]],codec,24))
    c.raises('超长不能静默截断',lambda:collate_completions(rows,answers,codec,10))
    v=torch.tensor([[1.,0.,0.,0.],[0.,0.,0.,0.]],dtype=dtype,requires_grad=True)
    adv=leave_one_out(v)
    c.close('沿同题组的LOO优势',adv[0],[1.,-1/3,-1/3,-1/3])
    c.close('零奖励组优势为零',adv[1],torch.zeros(4))
    c.ok('优势封存无图',not adv.requires_grad)

    cfg=TrainConfig(sft_steps=2,rollout_batches=2,inner_epochs=3)
    base=TinyLM(ModelConfig()).eval();reference=freeze_reference(base)
    policy=inject_lora(copy.deepcopy(base)).eval()
    c.ok('参考与策略独立存储',all(p.data_ptr()!=q.data_ptr() for p,q in zip(base.parameters(),reference.parameters())))
    c.ok('只训练Q_V的AB',all(p.requires_grad==is_adapter(n) for n,p in policy.named_parameters()))
    with torch.no_grad():
        c.close('初始LoRA保持基座输出',policy(batch['input_ids']),reference(batch['input_ids']),atol=0,rtol=0)
    rollout=prepare_reuse_batch(policy,reference,rows,codec,cfg,torch.Generator().manual_seed(600),0)
    c.ok('每问题四个独立槽保留',len(rollout['answers'])==8)
    c.ok('采样概率与重算一致',rollout['initial_recompute_error']<3e-5)
    c.ok('封存量无梯度图',all(not rollout[k].requires_grad for k in ('old_logps','old_distribution','advantages')))
    c.close('新批次ratio为1',batch_diagnostics(policy,rollout,codec,.2)['ratio_mean'],1.)
    c.close('新批次old_state_KL为0',batch_diagnostics(policy,rollout,codec,.2)['old_state_kl'],0.,atol=1e-7)
    b=copy.deepcopy(rollout);b['old_logps'][0]+=.1
    c.raises('拒绝旧分母被刷新',lambda:validate_reuse_batch(policy,b,codec,0))
    b=copy.deepcopy(rollout);b['advantages'][0]+=1
    c.raises('拒绝优势跨epoch重算',lambda:validate_reuse_batch(policy,b,codec,0))
    b=copy.deepcopy(rollout);b['answers'][0]=[EOS]
    c.raises('拒绝样本被替换',lambda:validate_reuse_batch(policy,b,codec,0))
    c.raises('拒绝跨采样批次复用',lambda:validate_reuse_batch(policy,rollout,codec,1))
    c.raises('测试集不能进入采样更新入口',lambda:prepare_reuse_batch(policy,reference,splits['test'][:1],codec,cfg,torch.Generator(),0))
    c.raises('参考未冻结拒绝',lambda:prepare_reuse_batch(policy,base,rows,codec,cfg,torch.Generator(),0))
    unrelated=copy.deepcopy(policy)
    with torch.no_grad(): next(p for p in unrelated.parameters() if p.requires_grad).add_(.1)
    c.raises('拒绝控制器外的策略更新',lambda:validate_reuse_batch(unrelated,rollout,codec,0))
    opt=torch.optim.Adam([p for p in policy.parameters() if p.requires_grad],lr=.003)
    badopt=torch.optim.Adam([next(p for p in policy.parameters() if p.requires_grad)],lr=.003)
    c.raises('拒绝漏掉adapter的优化器',lambda:reuse_step(policy,rollout,codec,badopt,cfg,'clip',0))
    before_ref=tensor_hash(reference.state_dict());before_base=tensor_hash(base_state(policy))
    frozen_old=rollout['old_logps'].clone();frozen_adv=rollout['advantages'].clone()
    for _ in range(3): reuse_step(policy,rollout,codec,opt,cfg,'clip',0)
    c.ok('同一批次实际完成三次更新',rollout['updates']==3)
    c.ok('达到次数上限关闭批次',rollout['closed'])
    c.close('复用期间旧log概率不变',rollout['old_logps'],frozen_old,atol=0,rtol=0)
    c.close('复用期间优势不变',rollout['advantages'],frozen_adv,atol=0,rtol=0)
    c.ok('冻结基座不变',tensor_hash(base_state(policy))==before_base)
    c.ok('固定参考不变',tensor_hash(reference.state_dict())==before_ref)
    c.ok('参考不接收梯度',all(p.grad is None for p in reference.parameters()))
    c.raises('禁止复用已关闭批次',lambda:reuse_step(policy,rollout,codec,opt,cfg,'clip',0))
    new_batch=prepare_reuse_batch(policy,reference,rows,codec,cfg,torch.Generator().manual_seed(60),1)
    c.close('重新采样后分母更新ratio重置1',batch_diagnostics(policy,new_batch,codec,.2)['ratio_mean'],1.)
    c.ok('新批次具有新批次ID',new_batch['batch_id']==1)
    # 非安全功能：测试中显式模拟一次已提交的大更新，检查stop只阻止后续步。
    with torch.no_grad():
        for name,p in policy.named_parameters():
            if name.endswith('lora_B'): p.add_(.2)
    new_batch['expected_policy_hash']=tensor_hash(policy.state_dict())
    tiny=TrainConfig(sft_steps=2,rollout_batches=2,inner_epochs=3,target_state_kl=1e-12)
    state_before=tensor_hash(policy.state_dict());optim_before=copy.deepcopy(opt.state_dict())
    stopped=reuse_step(policy,new_batch,codec,opt,tiny,'clip_stop',1)
    c.ok('超过KL阈值停止后续步',not stopped['updated'] and new_batch['closed'])
    c.ok('early_stop不回滚也不执行新更新',tensor_hash(policy.state_dict())==state_before)
    c.ok('early_stop不增加实际更新计数',new_batch['updates']==0)
    c.ok('early_stop时optimizer步数不推进',all(torch.equal(opt.state_dict()['state'][k]['step'],v['step'])
             for k,v in optim_before['state'].items()))
    with tempfile.TemporaryDirectory() as tmp:
        path=Path(tmp)/'adapter.pt';torch.save(adapter_snapshot(policy),path)
        restored=inject_lora(copy.deepcopy(base)).eval()
        load_adapter(restored,torch.load(path,weights_only=True))
        with torch.no_grad():
            c.close('磁盘adapter重载输出一致',policy(batch['input_ids']),restored(batch['input_ids']),atol=0,rtol=0)
    report=evaluate_policy(policy,splits['dev'][:2],codec,2,seed=60)
    c.ok('评测固定分母',report['tasks']==2 and report['sampled_total']==4)
    c.ok('精确成功概率范围',0<=report['exact_success_probability']<=1)
    c.raises('非法G=1配置拒绝',lambda:TrainConfig(group_size=1).validate())
    c.raises('零inner_epochs拒绝',lambda:TrainConfig(inner_epochs=0).validate())
    c.raises('负KL阈值拒绝',lambda:TrainConfig(target_state_kl=-1).validate())
    return {"passed":len(c.names),"names":c.names,"scope":"CPU数学与小型序列级actor训练；非完整PPO/LLM/GPU实测"}

In [24]:
check_report = run_checks()
print("固定检查通过:", check_report["passed"], "项")
print("范围:", check_report["scope"])
if RUN_DIRECTORY is not None:
    write_json(RUN_DIRECTORY/"checks.json",check_report)

固定检查通过: 90 项
范围: CPU数学与小型序列级actor训练；非完整PPO/LLM/GPU实测


## 16. 独立练习区

先从空白实现，再打开检查开关。不要直接转调本课完整参考函数，避免检查通过却没有练习实现。

**练习 A：PPO-Clip 损失**

输入当前与旧序列log概率、固定优势和epsilon，返回负的裁剪目标均值。确认四个正负优势边界，旧概率与优势不应得到梯度。

In [25]:
def exercise_checks_a(fn: Callable) -> dict:
    c=Checks();d=torch.float64
    n=torch.tensor([math.log(.6),math.log(1.5),math.log(.6),math.log(1.5)],dtype=d,requires_grad=True)
    o=torch.zeros(4,dtype=d,requires_grad=True);a=torch.tensor([1.,1.,-1.,-1.],dtype=d,requires_grad=True)
    got=fn(n,o,a,.2);want,_=clipped_surrogate(n,o,a,.2)
    c.close('A_目标',got,want)
    gn,go,ga=torch.autograd.grad(got,(n,o,a),allow_unused=True)
    c.close('A_四象限梯度',gn,[-.15,0.,0.,.375])
    c.ok('A_旧概率和优势无梯度',go is None and ga is None)
    c.raises('A_非法epsilon',lambda:fn(n,o,a,-.1))
    return {'passed':len(c.names),'names':c.names}

def exercise_checks_b(fn: Callable) -> dict:
    c=Checks();d=torch.float64
    n=torch.tensor([[math.log(1.1),math.log(1.1),500.]],dtype=d,requires_grad=True)
    o=torch.zeros_like(n,requires_grad=True);mask=torch.tensor([[True,True,False]])
    value=fn(n,o,mask)
    c.close('B_序列比是乘积',value,[1.21])
    gn,go=torch.autograd.grad(value.sum(),(n,o),allow_unused=True)
    c.close('B_只经过有效位置求梯度',gn,[[1.21,1.21,0.]])
    c.ok('B_旧概率停止梯度',go is None)
    c.raises('B_拒绝无有效token',lambda:fn(n,o,torch.zeros_like(mask)))
    return {'passed':len(c.names),'names':c.names}

def exercise_checks_c(fn: Callable) -> dict:
    c=Checks()
    c.ok('C_仍有预算可更新',fn(0,4,0.,.02)=='update')
    c.ok('C_恰到KL阈值仍允许',fn(1,4,.02,.02)=='update')
    c.ok('C_超阈值停止后续步',fn(1,4,.0201,.02)=='kl_stop')
    c.ok('C_复用次数耗尽',fn(4,4,0.,.02)=='reuse_exhausted')
    c.raises('C_拒绝NaN',lambda:fn(1,4,float('nan'),.02))
    c.raises('C_拒绝负计数',lambda:fn(-1,4,0.,.02))
    return {'passed':len(c.names),'names':c.names}

In [26]:
def clipped_loss_ex(new_logp, old_logp, advantage, epsilon):
    # TODO：概率比、固定优势、min、负均值。
    raise NotImplementedError("请独立实现练习A")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    print(exercise_checks_a(clipped_loss_ex))

### 练习 B：从逐 token log 概率得到完整序列比

输入 `[B,T]` 的新旧log概率和布尔mask，先屏蔽提示/PAD，再分别求和，返回 `[B]` 的概率比。不要对每个token的比值求平均。旧log概率需要停止梯度。

测试中故意在被屏蔽位置放一个很大的数；它不能影响结果或造成溢出。

In [27]:
def sequence_ratio_ex(new_token_logps, old_token_logps, mask):
    # TODO：mask、序列求和、旧概率停止梯度、exp。
    raise NotImplementedError("请独立实现练习B")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    print(exercise_checks_b(sequence_ratio_ex))

### 练习 C：决定是否还允许下一次局部更新

实现一个纯函数，返回以下之一：

`update`、`reuse_exhausted`、`kl_stop`。

先检查计数是否合法；达到复用上限先结束。剩余预算存在且KL**严格大于**阈值时停止。恰好等于阈值仍允许。拒绝NaN、负计数。

注意：这是下一次更新的门控，不回滚上一步，也不是硬约束。

In [28]:
def reuse_gate_ex(updates, max_reuses, state_kl, target_kl):
    # TODO：输入检查、次数上限、KL停止。
    raise NotImplementedError("请独立实现练习C")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    print(exercise_checks_c(reuse_gate_ex))

## 17. 参考答案

完成练习后再核对。下面的名称不会覆盖你自己的函数。

In [29]:
def reference_a(new_logp,old_logp,advantage,epsilon):
    require(0<epsilon<1,'epsilon非法')
    r=probability_ratio(new_logp,old_logp)
    a=advantage.detach()
    return -torch.minimum(r*a,r.clamp(1-epsilon,1+epsilon)*a).mean()

def reference_b(new_token_logps,old_token_logps,mask):
    require(new_token_logps.shape==old_token_logps.shape==mask.shape and mask.dtype==torch.bool,
            '形状/mask错误')
    require(bool((mask.sum(-1)>0).all()),'每条序列至少一个token')
    new=new_token_logps.masked_fill(~mask,0).sum(-1)
    old=old_token_logps.detach().masked_fill(~mask,0).sum(-1)
    return probability_ratio(new,old)

def reference_c(updates,max_reuses,state_kl,target_kl):
    require(type(updates)is int and updates>=0 and type(max_reuses)is int and max_reuses>0,
            '计数错误')
    require(math.isfinite(state_kl) and state_kl>=0 and math.isfinite(target_kl) and target_kl>0,
            'KL错误')
    if updates>=max_reuses:return 'reuse_exhausted'
    if state_kl>target_kl:return 'kl_stop'
    return 'update'

In [30]:
print("A:", exercise_checks_a(reference_a))
print("B:", exercise_checks_b(reference_b))
print("C:", exercise_checks_c(reference_c))

A: {'passed': 4, 'names': ['A_目标', 'A_四象限梯度', 'A_旧概率和优势无梯度', 'A_非法epsilon']}
B: {'passed': 4, 'names': ['B_序列比是乘积', 'B_只经过有效位置求梯度', 'B_旧概率停止梯度', 'B_拒绝无有效token']}
C: {'passed': 6, 'names': ['C_仍有预算可更新', 'C_恰到KL阈值仍允许', 'C_超阈值停止后续步', 'C_复用次数耗尽', 'C_拒绝NaN', 'C_拒绝负计数']}


## 18. 验收与进一步实验

完成本课后，应能独立解释：

1. 采样旧策略为什么既不等于当前策略，也不等于永久冻结的参考？
2. 为什么不能只写 `clip(ratio)*advantage`？
3. 裁剪后出现1.6的概率比，为什么不必然意味着实现错误？
4. 为什么同一batch可以有限复用，但不能随意混入另一轮旧数据？
5. 本课序列级裁剪，与逐token PPO、完整critic/GAE系统差在哪里？

继续实验时可以预先修改 `target_state_kl` 或复用次数，但要记录配置、实际更新数和采样成本。不要看着测试正确数反复选择参数，再把同一测试称为独立留出。

本课不包含价值网络、优势估计中的bootstrap、GAE、PPO value loss、AMP、分布式训练或生产后训练框架。CPU默认实验已执行；CUDA路径尚未实测。

## 资料与下一课

- [PPO 原论文：Schulman 等，2017](https://arxiv.org/abs/1707.06347)
- [官方 PPO-Clip 解释与边界](https://spinningup.openai.com/en/latest/algorithms/ppo.html)
- [PyTorch PPO 实现：旧概率、优势、KL停止](https://spinningup.openai.com/en/latest/_modules/spinup/algos/pytorch/ppo/ppo.html)
- [PyTorch detach：共享存储提醒](https://docs.pytorch.org/docs/2.10/generated/torch.Tensor.detach.html)
- [自动求导与评估模式](https://docs.pytorch.org/docs/2.10/notes/autograd.html)
- [可复现性边界](https://docs.pytorch.org/docs/2.10/notes/randomness.html)

**下一课：第61课，价值函数、Return与GAE。** 把终局奖励与每一步的价值估计连接起来，进一步区分actor目标、critic损失、终止与截断。